# Bronze Table Creation

In [4]:
customers_raw = spark.read.parquet("abfss://ecommercePro@onelake.dfs.fabric.microsoft.com/ecom_lakehouse.Lakehouse/Files/Bronze/customers.parquet")
order_items_raw = spark.read.parquet("abfss://ecommercePro@onelake.dfs.fabric.microsoft.com/ecom_lakehouse.Lakehouse/Files/Bronze/order_items.parquet")
orders_raw = spark.read.parquet("abfss://ecommercePro@onelake.dfs.fabric.microsoft.com/ecom_lakehouse.Lakehouse/Files/Bronze/orders.parquet")
products_raw = spark.read.parquet("abfss://ecommercePro@onelake.dfs.fabric.microsoft.com/ecom_lakehouse.Lakehouse/Files/Bronze/products.parquet")

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 7, Finished, Available, Finished, False)

In [7]:
display(customers_raw.limit (5))

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8adf01b9-d711-43eb-bf6c-eadbacdc7844)

In [19]:
customers_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_customers_raws")
order_items_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_order_items_raw")
orders_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_orders_raw")
products_raw.write.format("delta").mode("overwrite").saveAsTable("bronze_products_raw")


StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 22, Finished, Available, Finished, False)

In [18]:
%%sql
-- drop table bronze_orders

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 21, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

# Silver Table

### Silver order

In [16]:
%%sql
select * from bronze_orders_raw
limit 2

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 19, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 9 fields>

In [21]:
%%sql
CREATE OR REPLACE TABLE silver_orders AS

SELECT
    CAST(OrderID AS STRING) AS OrderID,
    CAST(CustomerID AS STRING) AS CustomerID,

    TO_DATE(REPLACE(OrderDate, '/', '-')) AS OrderDate,
    TO_DATE(REPLACE(ShippedDate, '/', '-')) AS ShippedDate,
    TO_DATE(REPLACE(DeliveredDate, '/', '-')) AS DeliveredDate,

    ROUND(CAST(Price AS DOUBLE), 2) AS Price,

    INITCAP(TRIM(OrderChannel)) AS OrderChannel,
    INITCAP(TRIM(Region)) AS Region,

    TRIM(ReferredBy) AS ReferredBy

FROM bronze_orders_raw;

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 24, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [23]:
%%sql
select * from silver_orders limit 2

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 26, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 9 fields>

### silver_customers

In [25]:
%%sql
select * from bronze_customers_raws limit 2

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 28, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 4 fields>

In [27]:
%%sql
CREATE OR REPLACE TABLE silver_customers AS

SELECT
    CAST(CustomerID AS STRING) AS CustomerID,

    INITCAP(TRIM(CustomerName)) AS CustomerName,

    CASE
        WHEN LOWER(IsPremiumCustomer) IN ('yes', 'y') THEN 'Yes'
        ELSE 'No'
    END AS IsPremiumCustomer,

    TO_DATE(SignupDate) AS SignupDate

FROM bronze_customers_raws

WHERE CustomerID IS NOT NULL;

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 30, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [29]:
%%sql
select * from silver_customers limit 2

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 32, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 4 fields>

### silver_products

In [32]:
%%sql
select * from bronze_products_raw limit 2

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 35, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 5 fields>

In [30]:
%%sql
CREATE OR REPLACE TABLE silver_products AS

SELECT
    CAST(ProductID AS STRING) AS ProductID,

    INITCAP(TRIM(Category)) AS Category,
    INITCAP(TRIM(Carrier)) AS Carrier,

    ROUND(CAST(ShipmentCost AS DOUBLE), 2) AS ShipmentCost,

    CASE
        WHEN LOWER(IsDiscounted) IN ('yes', 'y') THEN 'Yes'
        ELSE 'No'
    END AS IsDiscounted

FROM bronze_products_raw

WHERE ProductID IS NOT NULL;

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 33, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

### Silver_order_items

In [36]:
%%sql
select * from bronze_order_items_raw limit 2

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 39, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 3 fields>

In [38]:
%%sql
CREATE OR REPLACE TABLE silver_order_items AS

SELECT
    CAST(OrderID AS STRING) AS OrderID,
    CAST(ProductID AS STRING) AS ProductID,
    CAST(Quantity AS INT) AS Quantity

FROM bronze_order_items_raw

WHERE Quantity IS NOT NULL
  AND Quantity > 0;

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 41, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [40]:
%%sql
select * from silver_order_items limit 2

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 43, Finished, Available, Finished, False)

<Spark SQL result set with 2 rows and 3 fields>

### GOLD LAYER

In [43]:
%%sql
CREATE OR REPLACE TEMP VIEW orders_enriched AS

SELECT
    o.OrderID,
    o.CustomerID,
    o.OrderDate,
    o.ShippedDate,
    o.DeliveredDate,
    o.Price,
    o.Region,
    o.OrderChannel,
    o.ReferredBy,

    c.CustomerName,
    c.IsPremiumCustomer,
    c.SignupDate,

    i.ProductID,
    i.Quantity,

    p.Category,
    p.Carrier,
    p.ShipmentCost,
    p.IsDiscounted,

    DATEDIFF(o.DeliveredDate, o.ShippedDate) AS DeliveryDelay

FROM silver_orders o

LEFT JOIN silver_customers c
    ON o.CustomerID = c.CustomerID

LEFT JOIN silver_order_items i
    ON o.OrderID = i.OrderID

LEFT JOIN silver_products p
    ON i.ProductID = p.ProductID

order by DeliveryDelay desc

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 46, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [45]:
%%sql
select * from orders_enriched limit 5

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 48, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 19 fields>

In [46]:
%%sql

CREATE OR REPLACE TABLE gold_order_kpis AS

SELECT
    CustomerID,
    CustomerName,
    IsPremiumCustomer,
    Region,
    OrderChannel,

    COUNT(DISTINCT OrderID) AS TotalOrders,
    SUM(Quantity) AS TotalQuantity,
    SUM(Price) AS TotalRevenue,

    ROUND(AVG(DeliveryDelay), 1) AS AvgDeliveryDelayDays,

    COUNT(
        DISTINCT CASE
            WHEN DeliveryDelay > 3 THEN OrderID
        END
    ) AS DelayedOrders,

    ROUND(SUM(ShipmentCost), 2) AS TotalShipmentCost,

    ROUND(
        SUM(
            CASE
                WHEN IsDiscounted = 'Yes' THEN Price
                ELSE 0
            END
        ), 2
    ) AS RevenueFromDiscounted,

    MAX(DeliveryDelay) AS MaxDeliveryDelay

FROM orders_enriched

GROUP BY
    CustomerID,
    CustomerName,
    IsPremiumCustomer,
    Region,
    OrderChannel;

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 49, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [47]:
%%sql
select * from gold_order_kpis limit 5

StatementMeta(, daf8cda7-7d34-4330-a269-a73de3c10c0f, 50, Finished, Available, Finished, False)

<Spark SQL result set with 5 rows and 13 fields>